# Clustered Curve Beta CDF Model

This notebook consumes the CSV produced by `custpaydetails_clustered_cumulative_curves.sql`, fits Beta CDF expected cumulative burn curves, evaluates held-out items, and compares the Beta CDF approach to a pure linear spend model.

## Input and Parameterization

| Metric | Value |
| --- | --- |
| Input CSV | clustered_data_input_lincoln.csv |
| Cluster curve rows | 750 |
| Items | 54 |
| Train items | 43 |
| Test items | 11 |
| Global Beta alpha | 0.680031 |
| Global Beta beta | 0.984116 |
| Global train MSE | 0.03364064 |
| Proxy label CSV | clustered_curve_proxy_labels_lincoln.csv |
| Proxy-labeled held-out rows | 125 |

In [ ]:
import csv
from pathlib import Path

path = Path('clustered_data_input.csv')
if not path.exists():
    path = Path('custpaydetails_clustered_cumulative_curves.csv')
if not path.exists():
    path = Path('ci_item_clustered_cumulative_curves.csv')
with path.open(newline='', encoding='utf-8-sig') as handle:
    reader = csv.DictReader(handle)
    rows = list(reader)
print(path)
print(len(rows))
print(reader.fieldnames)

## Linear Spend Baseline

The pure linear model assumes cumulative spend should equal elapsed time:

```text
expected_cumulative_pct_linear = elapsed_pct
position_delta_linear = actual_cumulative_pct - elapsed_pct
```

This is a useful baseline because it is transparent and easy to explain. It is also too rigid: many items are naturally front-loaded or back-loaded, so a linear curve can systematically flag normal burn shapes as risk.

## Held-Out Model Performance

Errors are cumulative-spend-percent errors. MAE `0.15` means an average absolute error of about 15 percentage points of final item spend.

| Model | MAE | RMSE | Median AE | P90 AE | Bias |
| --- | --- | --- | --- | --- | --- |
| Linear cumulative spend | 0.1452 | 0.1887 | 0.1197 | 0.3006 | -0.0015 |
| Pooled empirical median curve | 0.1528 | 0.1995 | 0.1294 | 0.3427 | 0.0674 |
| Duration-bucket Beta CDF | 0.1587 | 0.2028 | 0.1371 | 0.3430 | 0.0772 |
| Global Beta CDF | 0.1641 | 0.2077 | 0.1547 | 0.3542 | 0.0875 |
| Cluster-count-bucket Beta CDF | 0.1657 | 0.2106 | 0.1473 | 0.3504 | 0.0930 |

## Beta CDF vs Linear: Improvement Summary

| Comparison | Value |
| --- | --- |
| MAE improvement vs linear | -9.27% |
| RMSE improvement vs linear | -7.49% |
| P90 absolute-error improvement vs linear | -14.11% |
| Linear bias | -0.0015 |
| Duration-bucket Beta bias | 0.0772 |

## Error Comparison Plot



## Bucketed Beta CDF Parameters

### Duration Buckets

| Duration bucket | alpha | beta |
| --- | --- | --- |
| 181-365d | 0.618273 | 0.912769 |
| 366-730d | 0.696058 | 0.931197 |
| <=180d | 1.436884 | 1.812340 |
| >730d | 0.570170 | 1.035673 |

### Cluster-Count Buckets

| Cluster-count bucket | alpha | beta |
| --- | --- | --- |
| 13+ clusters | 0.734824 | 1.053867 |
| 7-12 clusters | 0.514427 | 0.789108 |

## Curve Performance Plot



## Risk Signal Framing

For active items, the same fitted curve can produce budget-overrun and time-overrun risk signals. These are not final labels without a real authorized budget and schedule; they are position signals.

```text
expected_pct = model(elapsed_pct)
position_delta = actual_cumulative_pct - expected_pct

if position_delta > threshold:
    spending too quickly / budget-overrun risk

if position_delta < -threshold:
    spending too slowly / time-overrun risk
```

For budget overrun projection once a budget is available:

```text
projected_final_spend = actual_cumulative_spend / expected_pct
projected_overrun = projected_final_spend - authorized_budget
```

## Beta vs Linear Risk Signals on Held-Out Data

The table shows how many clustered observations would be flagged by each approach at several position-delta thresholds. `LinearOnly` rows are especially important: these are cases the linear model flags but the duration-bucket Beta curve treats as normal for the observed burn shape.

| Threshold | Signal | Beta count | Linear count | Both | Linear only | Beta only |
| --- | --- | --- | --- | --- | --- | --- |
| 0.10 | spending too quickly / budget-overrun risk | 19 | 35 | 19 | 16 | 0 |
| 0.10 | spending too slowly / time-overrun risk | 54 | 40 | 40 | 0 | 14 |
| 0.15 | spending too quickly / budget-overrun risk | 14 | 26 | 14 | 12 | 0 |
| 0.15 | spending too slowly / time-overrun risk | 43 | 24 | 24 | 0 | 19 |
| 0.25 | spending too quickly / budget-overrun risk | 5 | 14 | 5 | 9 | 0 |
| 0.25 | spending too slowly / time-overrun risk | 26 | 10 | 10 | 0 | 16 |

## Proxy ROC/AUC Setup

True ROC/AUC requires true binary outcome labels. This notebook now consumes retrospective proxy labels from `clustered_curve_proxy_labels.csv`, which is generated by the separate polynomial proxy-label notebook. The Beta CDF and linear models do not create those labels; they only score against them.

These ROC curves compare how well Beta and linear position scores recover the external proxy labels. They are useful for model behavior comparison, but they are not a substitute for true budget/schedule outcome labels.

## Proxy ROC/AUC Summary

| Signal | Model | AUC | Positive labels | Negative labels |
| --- | --- | --- | --- | --- |
| fast_spend_proxy | Duration-bucket Beta CDF | 1.0000 | 14 | 111 |
| fast_spend_proxy | Linear cumulative spend | 0.9994 | 14 | 111 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.9854 | 44 | 81 |
| slow_spend_proxy | Linear cumulative spend | 0.9812 | 44 | 81 |

## Threshold Sweep for Proxy Labels

| Signal | Model | Threshold | Flagged | TP | FP | Precision | Recall/TPR | FPR |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.05 | 26 | 14 | 12 | 0.538 | 1.000 | 0.108 |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.10 | 19 | 14 | 5 | 0.737 | 1.000 | 0.045 |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.15 | 14 | 14 | 0 | 1.000 | 1.000 | 0.000 |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.20 | 8 | 8 | 0 | 1.000 | 0.571 | 0.000 |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.25 | 5 | 5 | 0 | 1.000 | 0.357 | 0.000 |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.30 | 4 | 4 | 0 | 1.000 | 0.286 | 0.000 |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.40 | 3 | 3 | 0 | 1.000 | 0.214 | 0.000 |
| fast_spend_proxy | Linear cumulative spend | 0.05 | 42 | 14 | 28 | 0.333 | 1.000 | 0.252 |
| fast_spend_proxy | Linear cumulative spend | 0.10 | 35 | 14 | 21 | 0.400 | 1.000 | 0.189 |
| fast_spend_proxy | Linear cumulative spend | 0.15 | 26 | 14 | 12 | 0.538 | 1.000 | 0.108 |
| fast_spend_proxy | Linear cumulative spend | 0.20 | 17 | 14 | 3 | 0.824 | 1.000 | 0.027 |
| fast_spend_proxy | Linear cumulative spend | 0.25 | 14 | 13 | 1 | 0.929 | 0.929 | 0.009 |
| fast_spend_proxy | Linear cumulative spend | 0.30 | 8 | 8 | 0 | 1.000 | 0.571 | 0.000 |
| fast_spend_proxy | Linear cumulative spend | 0.40 | 4 | 4 | 0 | 1.000 | 0.286 | 0.000 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.05 | 68 | 44 | 24 | 0.647 | 1.000 | 0.296 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.10 | 54 | 42 | 12 | 0.778 | 0.955 | 0.148 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.15 | 43 | 39 | 4 | 0.907 | 0.886 | 0.049 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.20 | 36 | 36 | 0 | 1.000 | 0.818 | 0.000 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.25 | 26 | 26 | 0 | 1.000 | 0.591 | 0.000 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.30 | 14 | 14 | 0 | 1.000 | 0.318 | 0.000 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.40 | 4 | 4 | 0 | 1.000 | 0.091 | 0.000 |
| slow_spend_proxy | Linear cumulative spend | 0.05 | 55 | 43 | 12 | 0.782 | 0.977 | 0.148 |
| slow_spend_proxy | Linear cumulative spend | 0.10 | 40 | 36 | 4 | 0.900 | 0.818 | 0.049 |
| slow_spend_proxy | Linear cumulative spend | 0.15 | 24 | 24 | 0 | 1.000 | 0.545 | 0.000 |
| slow_spend_proxy | Linear cumulative spend | 0.20 | 19 | 19 | 0 | 1.000 | 0.432 | 0.000 |
| slow_spend_proxy | Linear cumulative spend | 0.25 | 10 | 10 | 0 | 1.000 | 0.227 | 0.000 |
| slow_spend_proxy | Linear cumulative spend | 0.30 | 5 | 5 | 0 | 1.000 | 0.114 | 0.000 |
| slow_spend_proxy | Linear cumulative spend | 0.40 | 0 | 0 | 0 | 0.000 | 0.000 | 0.000 |

## Proxy ROC Curves



## Residual Distribution Plot

Residual is `actual cumulative pct - expected cumulative pct`. Positive residuals indicate spending ahead of expected position; negative residuals indicate spending behind expected position.



## Recommendations

Use the duration-bucket Beta CDF as the first expected-position model and keep the linear model as a transparent benchmark. For production alerting:

- Use Beta CDF `position_delta` for primary spend-too-fast and spend-too-slow signals.
- Show the linear delta as a secondary reference because users understand it.
- Alert only when position delta exceeds a threshold and remains elevated across updates.
- For budget-overrun detection, combine position delta with authorized budget and projected final spend.
- For time-overrun detection, combine slow-spend position delta with schedule metadata; slow spending can mean delay, scope removal, or inactive work.